# General Pretraining — Consolidated Research Record & Presentation Figures (v3)

**One Jupyter Notebook, single upload.** Uses previously saved HPC output, and NEVER re-runs the full unique/duplicate analysis, checkpoint inference, benchmark fitting, or conformer generation.

**Figure naming policy:** presentation charts use **General Pretraining / Frozen Encoder / Model**, not the original model's proper name. This notebook re-plots charts from saved numeric evidence rather than showing older presentation images with old text. Original HPC PNGs/CSVs remain untouched.

**Sequence:** (0) existing outputs → **(1) historical and matched 10K train/validation losses — research record only, not for presentation** → (2) masked atom and 3D recovery → (3) atom-level accuracy → (4) PCA → (5) independent property benchmarks → (6) saved unique/dedup audits.

**Run:** Kernel → Restart & Run All, using the existing Python kernel. Note: historical training history is archived in the supplied source notebook; plotting it is **not** retraining.


## 0. Configure existing HPC result directories (read-only)

In [ ]:
from pathlib import Path
import json, sys, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Change these ONLY if HPC mounted a different location.
GENERAL_CANDIDATES = [
    '/project/yli4/ypan/heart_evaluation/general_audit_10k_v2',
    '/mnt/project/yli4/ypan/heart_evaluation/general_audit_10k_v2',
    '/home/ypan/heart_liver/model_train/heart_evaluation/general_audit_10k_v2',
    '/scratch/ypan/heart_liver/model_train/heart_evaluation/general_audit_10k_v2',
]
DUPLICATE_CANDIDATES = [
    '/home/ypan/heart_liver/model_train/heart_evaluation/duplicate_impact_expanded_evidence',
    '/scratch/ypan/heart_liver/model_train/heart_evaluation/duplicate_impact_expanded_evidence',
    '/project/yli4/ypan/heart_evaluation/duplicate_impact_expanded_evidence',
    '/mnt/project/yli4/ypan/heart_evaluation/duplicate_impact_expanded_evidence',
]
def first_existing(paths):
    return next((Path(x) for x in paths if Path(x).is_dir()), None)
GENERAL = first_existing(GENERAL_CANDIDATES)
DUPLICATE = first_existing(DUPLICATE_CANDIDATES)
# Manually override GENERAL/DUPLICATE above if the auto-discovery did not find them.
print('General evaluation output:', GENERAL or 'NOT FOUND: set GENERAL = Path(...)')
print('Unique/dedup audit output:', DUPLICATE or 'NOT FOUND: set DUPLICATE = Path(...)')

def show_image(path, width=1050):
    p=Path(path)
    if p.is_file():
        display(Markdown(f'**Saved figure:** `{p.name}`'))
        display(Image(filename=str(p),width=width))
        return True
    return False

def find_images(root, include=(), exclude=()):
    if root is None: return []
    inc=[str(s).lower() for s in include]
    exc=[str(s).lower() for s in exclude]
    return sorted((p for p in root.rglob('*.png') if
                   all(s in p.name.lower() for s in inc) and
                   not any(s in p.name.lower() for s in exc)),key=lambda p:(len(p.parts),p.name))

def show_matches(root, include=(), exclude=(), max_figs=3):
    matches=find_images(root,include,exclude)
    if not matches:
        print('Saved figure not found:', ' / '.join(include), '\n  Root:',root)
        return []
    for p in matches[:max_figs]:show_image(p)
    if len(matches)>max_figs:print('Additional matching figures:',len(matches)-max_figs)
    return matches

def find_file(root,filename):
    if root is None:return None
    found=sorted(root.rglob(filename),key=lambda p:len(p.parts))
    return found[0] if found else None

def read_csv_or_none(p):
    return pd.read_csv(p) if p is not None and Path(p).is_file() else None

def section(msg):display(Markdown('**Interpretation:** '+msg))

print('Kernel:',sys.executable)
print('All following cells are cached-results-only; no GPU required.')
# Retain old cached files as source data, but avoid outdated text in displayed presentation figures.
# All regenerated slide figures are written to a separate directory (NOT the source dirs).
from datetime import datetime
FIG_OUTPUT = Path.cwd()/'general_pretraining_figures_v3'
FIG_OUTPUT.mkdir(parents=True,exist_ok=True)
SHOW_LEGACY_PNG = False  # Old cached PNGs may still carry previous title/model labels.
def export_fig(fig,stem):
    target=FIG_OUTPUT/(stem+'.png')
    fig.savefig(target,dpi=190,bbox_inches='tight')
    print('New figure:',target)
    plt.show()

def safe_metrics(x):
    return x if isinstance(x,dict) else {}


## 1. Historical training and validation losses — **RESEARCH RECORD ONLY**

This is the **original optimization history** captured in the prior results Notebook (archived observation series, updates 510,100–1,000,000). It is kept at the start for completeness, but intentionally **excluded from slide figure exports**. The historical train and validation loss series have potentially different logging/augmentation aggregation. This is not a definitive overfitting test.


In [ ]:
import io, zlib, base64
# Archived original training/validation log observations; no model computation.
_HIST_TRAIN_B85 = 'c-n=%-L57(j2(6#Aa7y>694%$gTdGrL1Itf8Rz+}Q<4h3dl$KG8V6?+Dv6@VwMa_e|M*}3{a^mq|My@2^*{gHfBcvK_uv2HKmN;q|JVQcfBxHl{g;3Jw|~3;`EURJ@Bj0^{>$^nfBX0U@t^<mU;pDj|L_0*zdirs|NK96jycAE8UL5d`X}Ajzhm-0|7uLOM}m$7{T30Kj|guOj$a}kNsoxsA{qRM=$})GKfl8>9|=2y_{Si$zvE{R{ykgPM<VL=%=AYj&v){-$na;BrM6mR$|KUK)}HT7`mvt)5~)@DPDRR5uJ)%kqnz2=50THgT;0ynSY}|6qg=7yBEj*G2#tD_A<uMaLcI{xCg;zBh@07NQ=ranx5<&#rf}9w)-34E_9&P2_lK1BDA9cAOgG_*cvNP24<b*viA?2G-ch(-(6h@d=zQ?i?C-<8%;!-X8(hs*?1zZW{hg@&%ARUA0`~Yo8I4A^p-7|G<Kcv$o0O|p`!MUVN0G*s;#smNo{QBVI(|jG+`>CZp!gz<$;y0mtD|<=+o>xyStWRSl#xa1)$IK3YlR=rq*t^0u^xZardIFI_Dt$ckUHy$I?|oEthH*dHQ9icZ4(AeYs~hJ!Z2W(K%Z_3&xct(%2Ay;jPKTHm*O#P%Vrn%LG%Ij;x$S*YH>nG$Qf$Iv2I-<iu3DjKjvbwM`=L9QJh1c;Mq*&AAuN+6MRfE8mlGRgFwYb6T>8*^@cQta=xPoh=f6dj3$g7pB~YK%QM?-^v2Kb<fyf>PkYWEKEpl?CF#d^;=`<yYaD2_MFd)8Dg-=fXq&U~wmSBApv`8Z+700TfwjNCBY)yr8WZ778W4NYOUU?lnA+P+4Iw)#qS0XleI$O`?FVU~@e|j^-$$7`KDOf}T|QP*P?~idyEB=PGSH+(WN*+*$UYFjhLbcK_;p1TfjD{z8JG5S30b$IXi&Iq)g>Z_bO~9xi8KKaBqD?+iMamCF>JzC`_eHlRZG@*r>!^YScB{}2Ra_qxP)g29u0)p=kz6GAE6<)1VNp2T<qwCr9ME%^ciLmhz-PLdg)jhr06d?_O(5n@QBHPo>XSD=KUcd=$&3dR%`AhWF`2eOUO!2KmxO0)bO#po~Z5mSezvMj*dE6be8-VBa7cSkd*K!4BeMX!(g^!(Z<`}Q4E@;dY{lB4Fn_TGO{oApP6)w?5O`TvYLg`%gCz98UsK>CU_!Ef@XtKGK1UG(6RSJjoF@H*wL}`As;XXT{`x)qT?xFJ~k`FPu{tNect~fWM9zfC1hVvW5*yznFMl&C&&X1l8r~+>**NTGu4<VnMcuJgkuOVBcsysUq<%zp(~<@o=_O{>Mtdu(r3Jstc-_|C_PD8tM*RirDV-01Y=ObJOaIxjLH*#DH%%v7SkY(NQY4HGP1@J5=Jt<l&p5h{8BQjMF{yG{)>`*OqV-Kw)dGw6Unw-cr=mB_jej>yL}vw27+xL#{+;{eks|kd^W$7j6QhkGuM}q{T5(<);l4#<td0t@zi><Ec-*&O!j=pQS{&$C^Ya)l&NMK&{U{#3CC3EhjM?ec|KN4jdITkqRu01<&xIvh{sf3LRMx$xvP&I)l!rH+VPSvBcrU46LaSg9)XEa1b38dgJ0%`?A}qNiMo4}SrMFPw&p|IeRZ7d_$H3V_|J#3hny^(X7`e1LU()_+2<BMID%hBR&C+S$D~}1D9X3;OUQl?awQz$W0ruS;}SCZjaGcvvmhPtoa(GS+&LXeC>_DdjZ@z%Qh}DOHU4_1TPt^CD^{<)%3V;PWF5bZtny9Pgt6mz410u^kIf1zvM~%E*4XX+km3(nJ<8GkrDR`+Y*G711ZEQ0vH`A7K7t+NFQ4EvUqaT6!4NmO)E@5GN=L}flHHoj``$!vXu}3Q9%#i0bsqzfNB)bEeSmC!8CeM?J3@BCe+Z(W{1UPcS9%7q#_eyrzI=@011xSl^H+)N191LTA^S)^^2^7lRPE(s^x*@ad^ji7`*R#=GD62G5Rzt-S3H}XP4)4B{g;nf2}LqWxBrS{Bl5sJ>G;@Qk-U7Y$|J~VaP31Hq+?@`YGnv2p~*|ezW6`y<ybw}BjIo+0V1KiRIEBZ(s8lv&t*;&6=V_$KQb9%aG8Q4lyq$DuBh<``~1jk0K)b=GP@wQ2gx5!<x6waDEHVK%_b8OTZGUyUo!UT-_CdBhgQA}judX<OUGtC7e@hAsG$sxAyA@plkh-E9^nZ-!;LqP{S~gu*iR)-7<+?D#XfiEd<S}i%f#qobH>=~4V0-6?r_)kvo9Abaf%2e4AOD2vpvEnSc(V&z9JBLi5S9fBHqxxEz#KQae|U)6JIj+;W3^eE*1OoDN~=-Xy<CT@48g%>#QFqsZDO?s7)2o8&Sh^@p>+2%}2B3r61Lf=WgS-fzI;fVs(Z`E*B%OaH3o;_BGoFzlk-fJ$7w4iWy?L@)pNRbF(Jxb!_$}VkHk2Z}`HQtj~tEZ-4z#F)D+LGxU=#P;r-vHMVl8*w-mLK=D$sFPJ{^*+dWMzr2A6+gL6c`<m^sxJ$<9BcSdik6bd=EGQWIudY0Hw&=34<_txn+_Q@>9rF^dL^L926sX)Bh!5mMTv(D|$jk=X?lA<eCL7J}?YU&^Ymg)1kVR#{!<<>$+xEwLx!9Le-|NvaYs21=Oh@MuY<~#41byjP<q>o!LdQA``4#PIWBYXIPd@hHk@e+cW!TjT-L1}vzB38Ab)8Khd(26vfC`&VtxaWnFP#lz*eF70lbzY;R4*H=Q~Nrb0?wH(4Upqesx^DmM$}ApE1#N1(DNCAmQS$7L&-oENXhFx9=t`ML(r}?wh6UfHumM4&i?R35mhfA``|bhu}9(B;vT23-1yq#zE}|#R&J6ANB!JDgyZ2j5IIq<Gc-p+1~e9F!q_v{zv$S9+JsSxR7FtOR)NKGg|#siuc7a@B?)v)yUN&^>rXQFHIa^uowK2xJ9u!qzJ;;!fT;kiuL@HLss@OOc5RB=Unoj~t7bzAx4qgxtFc*1ob9v5Uhg0<ewiS!s@C-%?4aHdv9o<v+v{`qETBiSL~Rc0&|G~Y!mdtm5JvvU1>jJ9DVUaruc?NeVv2Uvu;cf<PJnN7eF<3Yqhr)|bgU#M%1dhy#4y3APNmT$z|SbPZY|}p$G;fZm$SeOOe0XLS6>EJ(i+W%-yYk~2@zGo&JInk=W@<z3Nr1)ZuVti&4L^qkjugva+Hm3w^r<BVO*3F?ImGbB0nSuAru>oQn@x7m#}Tc2DDonhCrmfBy1KT%m!@iu%;+>BrXU0%waso!!8G#zqMk^Wj!J_oIR#uvq4?kYHbQ2do~+5yZydRL(Mrh9q?$IHR{^BZz<Swe2hk4ld;Rd>TIyR6zumzt|<{Chhc9)tPBlC3Sg#D;|=bLuQ#BI)Y#=<WhTogkz+C*a4I}eveB3tpR%>W4BVS>1H6twz7$L<HfS#et2c-nPkYRo1Nu=;<kIbOurFqs^Vvk1aJo|08#kx4mxW0^h#&-R((&7E{g;J(?ek75n>d24CIJmf5^-(Q27#TqyiBatOUJ~H{VT&2T`Jans6n@P6qKs9w=7m`?yZZ>GGc0?T)Wvo=*q0I+&$a2F1F+0OUJbGVJ7f1ib%U%HumXFX)nj|im}~VLNMY325*O|y^O4xN<+yuN41xdarqfoxUsh?R{OZOD#oSPzLe}!<ViW2V}IM0#kM89WwDRr;kiZXV0Kp&joEI^g&=-tp?E=tY7iEVXB#LXWi2P$_LKu|QBG9gv4o7*UQSkniMKLFA6RZL`Xyy`1W$uVI~#|4M+rO%bh!RkZR~?y_&`0NJ_ya$M*|i1F8WK$$}RGjmQ^<A9V^=g<-<`#^brW}`ACEs{pDpPmiF1Wg|niUmr<#?$@Pkk{-@M)xgSylvixmUtelw7#&EZ>yv4D~hxtp)YVA{_+Ko1xCfsHs6rP=+Q*dfP*g0i{+QOX-&$j8Q+p48#{GjidbkXe5KEq(<JQ0-emzim)6vkHjQnM0N`b*7bdB_+h!JHj#b4ShYq8opERwM(u%9one2VStleYsiHSL7nl<X&o4A2Vs7!X~vYkUhB6?BShm*2w1A<x9=9{E#-Ld*YqKq=G9$SE&tDIKBm!$y^P8$Px5+4V3Wou4nOVqYYFi@xQp4e|T<d!rAs>mz&MsQ?aLz;wFu?>=R;&N^)*Z3Gyh;9J(ljFw6}m$K3}OnSJ%h9k$iEH!k=1Fc-Mwu4nO_AvYLAI(Bxf-vc^shw<zsXR|W6n9WSK+lUT?ZU1z+*;k7F9QeTDEVME0ec&%S>ps-z_9!j_kZ^ywS?!Z^zy)QK3p>h9j1>|_vy^6sw&GIjLAKYx%*-z{`_lda(j7B90VJVr<7H+aSljrKn7<|HPiGsKn9V3$X^bC=&EFdsqf_kpOp_kp!6OC?WbyK{%B*8RUmKT}{a7*mxjiuliXJg1@G3Do1eW_d49HSkQa1f#d!F|oV=A19fx2o;hdVd}(es#zY>B!|%{FS9=#&N_fDkVyqe>^2l6}qf93E0=OJpc6#N}k4O4&O|0nP0tWu?0g9#C;zy==$-cv)HHyMh_Hw_rh}AUaZZMGU3&f+24Rj<UFVjD-`S`H)7p=R1P!RtTQ5wqCZUf(=(t6bmsFPeVFjyKSYRWQSPCrDQ++7ya2)5IRtTD|9voWGhY~{0c>N-sp;DXPZ^4i26>pLkN3}1|S^2-`2{`cfJ4u{V?0h$3Ai3-XJRx!VRfLh!KP_^puy7eNE-KK~ognUp`i2qL+`=ihKE3J;;|wpg$Q-Tsrm{nEhFgPw2Cq&`EVhvHPWC9~f^h#}#FE!>w+tx^s0D-7gvYl2F-`O_aHc%f>z(+&&%A_}b&X0&4b?jnNN>9fD#<z>Ii=p-fzKWbBR-s5%%g8~eS5XURDb-n?9_QGOB}gdv`3x!AVaj*A@$L;NSb#`?zrW@EG`LS+KSl(2I#$3!`}9E7R_IT~EMeSTsxmK_<}=9c5DErEW1$dL!k0duxF>L_-0C}-WPbcj^WiFid$0&X%VBfObYN}uu0_9%C1<pgv7OwW8!>TC$vD-tN#ntu^8|CkH?jAK7U*q#s3L?|MOko{jq_N5li{T+d7xs<FXQ%QzEX}*-KnM>3e=Bt)$HhM3bM-3MDf_TCM%>?{lGD;&POYH}@t4vntH!dOjp%nUY;@JT;&zFzY5)EfKRx1s#J?puA?8`lEzn+(m9g+NnX!6prDWw^RKu>x3*r(q1cMSScfq8@!NE-6xV^$i14n&7jVf`^*J~k_Km6wmz+%OgU+e^rPY|#(ps?}%t*pqr~30Eyq6QwhgM90uZz2Y;7O_aLNj*uP4-x0Da(h#!aoJ^7Xx`gcb!2H-ka~M4?A^YTp+xjh`SN8!jSLrZClb4YFaOyvOk2ADM7wExU5i(8x0dM_7EAG%(UQSkf7XRQ4^hdrTV}NXG#zYAD^07K=m@grV(jP<<$L@!;@hbZmh9Q3(j%+ESsUqyd5ea8<3E3Bh5Pd8%CsKY=jhl}1N=8998&7*DtI+V%@Om71$qZynnE=2&SIK(BvqUCHS1b{#x9~YVOQeZn&vciF02;JJ6u7`8V&HhJzkBVD?<m)9KgxOh&kJx%B>SS_a|^G;JWsZ0iOkwqp7XIp2qa*Z$W<=?UeJTFmx$CJ`>199ga|)Ic~IOEu{z`T1k5GUDEBe{B@%!VERm+H`=G1m24jyBmIy)#OQZ&1cLP2-*%6U=HsIxCR=9;F(g84ejDZa9G<ky;`8gxhFo$DxlmPcUmp|J)z>+{6w@ZXT^j;zjOt_;oncJ-uR(R&wKA`JgQ*i(omq-BLzC>zxd+e#JlAS&5SlRJVLePl`L+366xaty7fJ~Q&HCDNOp*V>0OC%cSGM`<PY;foH96I2KE-(8!igy%3&Ru`{+@D@Sw}|urYDs8w;<qWN^?3~&`jhhJFKTT=f97fO<AAhvvS$zjY4Ig70oq$4AO*H0I;%b35zw2mL`qV2bz%nX+8ofFB@z(VSfU+lt7MO%_-w3tA7v*{@m?ZaNB9%TUQQ-uuyd`oaw3@<bo}n{D0`su<BdN!hw!bFSs5EHEt~TL_-vdU1R+bu%g&lZV6clilwFSy27%8dW{q=-z@uc7w%8Vx11>rzD8Mffs#8yAy4kqfvq#yqd>rkj!ef*eC}gAEV+?<l%kpCz6E!A0&r$rzLO!-}jvC?~?=lA=jxh%#-!U&c`{Ee;)X(Np+1A_~9r2C5Aa5Yn<51^lOm}be9EotvoCAx@dd+9LUA@e<1UIqwiDz?4pEO+(O-1L57z)1g<z|&boihLmn6uIB`OM4AJ~$Q0vxPL_E)#*g_Yz?Q<mSuFKIXDX?IcB>aK6m!x5SKR3(1jZnG~R?utc&UiQ6W5z-{KnY|n?7;9Op2My2Lh5QLhnC^a<M5D1>^W$yKyvX_}PnoDH;(&+a1IGNB5zRXN2)HsF9$a?t26oFPL4SP5d<gJ)V1*WG!AU35@?ukPkGdud%QNxMjwqIuUl|0&RK?!01sSV6Ms#9s=^A4LYH{*(6Pc@KdUm^};)l1H}B4<+@?VfG4fw;=FhD03KVU1rn(=~>AF+rGnnj$<mUxM}(M*PvPIbeCS<7am&36GHv7{k3uDjbjMRF87((mSWS)#3urt3#MO-C*eS2Iigs&@r^Dm#oRv_PQNIJ1#XQV%hU$XtP|<%g{=mH$$Ksdl_0ih{50*jnlmg3apmKEt~_L10=F$PDJ5m5<JhzD8n>Up!-Qhv}vZnSoHF<-%~k$BLkw6>|E}%i8MN#JGaP0xe^+Ld^-))gS$i;m%9y$-s0J;szmhiGp>RNz5Hxe`X|iK=;dc6g{C=>xzL;lBhe(34yLYpc2@LP^(?<eQFKOVHrhSL@G>;2juFj3g(RAZ939a>SyrNnd?CFY?PI#fUWlPy2<6F%6oEb>L#UfxjwbbN19a*vk;Zh72oF>PrI({wEkb?E){@{RS-AZI!@4iAqAUloHLhc8jvcxVjvCxtkwo;0q)Ccx_f7&6x+H|6D+#t-X_8{+6ul&kN>3+-_&SNhgC+r8D!gX)JbemqIbD)A@*}=J=^aw4H{h7mC23!yjC?jg1;!C=yzV(hYUmhg1ICU-KvQ@rTD{LNMJttJ6Z&=|81sy=h`<wKiXbXk6VUF88lQV?+hFR3ED=Q}AJEZxjdTyuBX0uQHU)3c!e&Ku8QO@C1F~_UYnf2IgFv%&SobAwQVYjzI7~UxOVG-x#8+1AN4pdSp;`t7L&O(?m@Qp?_ND!CrfX=I|0<w;DemQGvxJQ=KP#7<qn^n*hdCkOPORJf%8C6bU{VTlgW0Hd=fUaHv++?roAew*Tt1`3cAJU7G>?Js9*+r3n+KsR6c0u|j$3;6xMUtpMr<EM9)<uE565$94EL@mo0p^ZTxR1Gw!h_<o>2wncsAPIj)N!Hqio~mDbN<CB6!Nv>a@WFYj`@URPcoPMxKeHCR2?Rc6i{IoK;wYiBP(oFE<+>v}zw#CPB1{Nt->)JzGQkjmhDH0v-@bI29g<$}rEyo$k3@ZZ@*?3e2_0c9wj}Stp_>^UT~tv+Ym$lCw|9KKCervc1d+UEw8XpU~%sOw^mgLiKddT!c|_f(0gRSQHU;zT|BD_T2wG2=s_JBCRX|5C;o3-OJ6sV(UgW+;X$M(MnKr%1thPpAi-4{#6kSX8nm4;%h}XplLKBXD$h?`?9moBfb6qrDtFM@`?~#v|+f=(X(?#OcCZ|YbeBKaBPVBvIRn@Yzb%UbwE1?<r9W^hgk)F*;(m0Tu}Ccy;1G4Cx=+U>z0L0&iInEVdZi5gzzXDkb5@yva_!boiG!G3V9r`G|3?-eZir4x(2s79Ch_@z^npCLO6Z-*_Z0han4zv__Iezh&OREs^4&8%gs8iaH4NE6ynYn`h*M^t1dnJidB0mGU|-Y>A{cz8HBw2tXi@RM$lh|AiyC*k?)sLLxr==GQ0^gbBhGx`!c%v&YtcZXp3Y{<XmJnK<sFD)*koFb_xotG8LU?x$Nv~Do42!mLTNPvk$ZU=l1j?lJ!RK#I#&`Hnj2zLfyLxxkU)%0OjRpWrUE9pB)L9R}eB3z@@95?cAJD7tfC3643+U(SYe^5T2l;M3_+Hx-f&(oUkBrESH=$*2D$?iLfIFcB$Eix&8AD$Rm<~J<H3?q#WcjvoAD$%vzdAcGe>dx-aA<W)&C|QB<A~F=*D8mwo0Ux68P^?9=gYA1Dd!KnDyTI#Jp|E-@Qi?vOG<^rC{AuSHSuLDUdp6m67yj770>HK9GVylflP%ga6vNXUaE3_>t5hG66JvJbu71Alqhrx)DcG3384D{I`O1>7OarpfrpR*DY@M|_sXvXe{VFpgbT_L;cYk>RCfHD2@vBpfH#(Xum$BYrL~FOzbUVN;`YwCuR2n5dCfqDeCzKNw9mVW(Cl1SOm$6gB%x7y^2im3=7gZu4bjjb&d});QDbvW(lu<;0^<jp7cvN=DTz>8fT2Ya~XuiK@Mlj_9{!K#GzunCTeEvFn&<7xmJz8Vu@S#6NUMjY#fv6=7t3d6`z`qQj9z>8Pp0@q{|EIj(y}voYPXqB#-i(n+XIP$$-I*VVYugib{PPiGg-9yg}5vE6+}bOv5@g2rp5fjV5ZshjPisMHl7&Uc#9&N1$nmz7{bNm0mCGJuqoAXHEaEP~>hGFB~j+421yrg}YR5JtF0Sx^I}GFB~j5O*I$c9axV4m?VtD_Hth)$GfjWY)Q*u?L}g&F8>NO?r=Iiy%C%mz#Zg)3Za;G=Us%Dqh>uaS3nDtfJ`pQnT9k>p);m6-P7epu*lDg-}5|YIcsZD%?bEGPuXHsKdjmO*A{UppACVE_NM_dyBe~+0mR4+(ly;o((J46lGS`tf|QFU9{$Q$%aQ35J6c2R;51R23>0QW!tV>ga8}Wmz$L+SY4o#b48BbVPlxS-0UL(;<4A$neFyZo@mK2P+mx5!UG=-PB?46-0W*Xmd|O9#^D~b4Mo{z4MSyr4b)$w5l|u_G|GWRCX6vjC=;p4O+ef`lmbmIF;GZY)zNh5<T7AL%tS!i#z5%EMw1KM0nY{@gR;ShKHF%#!d9KV+^kFh+NcrkU2}tQw?)~A29i7*Z3EULZ4Pt|!)DDM9#?4-(q5Yk*E0qAew&IMl)V+RGM{Xhn03<)RNl6?V#cM(5$w7%ns5_eY9>{zGGgR8j0A1~DGG|#)yy^wvzMFIN09>LPK{tX%q}+@^zoe$k4WQl&qni4<;K@2SGn^ep4oGV-m2M$h(3=f8Y_Ym#s>AzjzwUO+%7pA`myM}MK3wyQoGV*Y&Nj8mz`0C)l3)_I!pwMC?Y)8mCiPSHt{;>?Qmt%IEIZE4D&P7fk=jFO?~f*v~jz~Np<|}ehA_PQ*FS&5osvIIk1Xt_vL3F_Ojbft?Oyri3PK!3*<N{n=k|zp{s3X44wB|J1g@BW{Td>Ff)`Vw61z~4ATN+0j=w4JZtvaG2Au#M44yvpH<)ZUAr`A)DUNXQM9k~>I%<@8)`JLm!ti*s5Q*~PyjAyj=@-9ztFWcUJ*j69ScTe(4s-Q2QV~`g6#7g1`kL$)=HrCki8tO#z+aTN?KwtmW|InK`$9b*KE3M&Wf_BO&*SLIGSaa(KWl`<g?8$MdMN;YjCsAs*i!-vX6;6aeTmd=7XErK7QM~>e)6ZABqZW{#MV*GvqBrJJSt>e|#h=ANw3A{Q8`5(UTAq@Ry{Or^sKD#-%aL8jHQtH53*7C23UaB~SvwUy?@UiU|5#e@R+N^_&RND<_7FCP9flrv{51t0P#5aB4bow|DV|q#ea2Tn`Tf`J5-Tmdnw~GsLim$zO(6AG4039kq1~ZF_%b$D=e};kZJE@ywgn##aQ^cwDg06<C~f={gpUh(gmALs?W8>~kehLeV>hcJ!j~3WRHyE#DHb#^cI}XS;SlY->&-{dF~zI&<~LF3%`|vTLqQv&}}rZUbepy(4JHEf_)n(-5>fT@z^CZ*ybq{$GMt;Vd_p7W1yV@ff!y1QdUj(7u%G6Mo3MZ23MS&M43AF3?<DAue>oB8}-DSmI5u&E}Kt*`UjPUSlBQ>M>E-)}xGQn@xJabh(E>s|@UOUxqfTjEe`>Z3H}u(OK?O=JdnR79}b5@-nnKbpFNAK62syGPD{Mrqqo8EIJjjMNiBk(oq`QFGVYl{AFm>7Dm_IxD6o&ywDdfKYMiYp^_M#zkL=HL&}8v(R~a>UL#(5Mzx3|uMr~<_KcAr$j8gh$`cf`!MNKbV-iZ=MaRzeA&V)f#u+U;JEK@-PG=8ile5`%t(bxEcFb%z>*FV*<7OLg2TjBig9wr1pr0NgetV}NLB~BvQFbB-)_8&(0sEkU21QNQX1k4Gc%d(N!WGr*>{vfDJY|fX0xL8q{=;rRvTo58&yMLcaJ9jWTF+;saR|qj1l~;#fdDK7D^>)xZ%0!)yT?!}#qM^V5Uf}U#HT_MrFlaKG)JANJ`qYV)F~sh5{H(3qm`#D9G8|=a3<6c<BIOa*>Qm_Dcc@fJG0#)2J>L?lCowLg6cO$h$miBRy*}W%083dNBw5dznLQ>7#0X;$6;lbA3Lq{8Ko<moe%k};MHAo*VnjNL0nQ+IxrXLG+gmq4b@M>u~TqVdx);B@oZthmeIJ3tUSv>jbU%B7}#n<KoTGV1B9?YZ7DuXDLR|op9y$;BH>J4I_6bVOJw5^PHf1gd+Efngk^_FfvB~6rtyBr8s~IIad_GjHYr3Vq9-}(_N8+Ub2OoBd*mF1PIDek_8F)4Hs*lI@f>9++te?IA;slkWS=fuVt*#Sk;buu?5bsZ2XhXzJ=xtg+o&ifA)J?OOXEYTa|&`AIThJtc8`E;ZkQ9su{@0IJ77L%HD{t6aneA=;3Sx%Ng}8hVmt~=j3>b^@+3pvCodx_4T=Jxo}{eRe~w|>(YTk)6S0D1>=Kwk1j$Rt%1vfiaY(iXAa8?XSaHajksaN0qArNMjI2V6dH7i8BM%?@K0drhDY4fbLlAz>myMN3B&AHx9IbYA>=?ZS3jZlToWVyLHFcmQb?QWxrj1@VI!lE-Hx;(o<y#}GZB1djQmR9D{nD{AA()!b|JkTBQpaj-!bKY)9-GF{cXdsS$I-rY>;tyS9>nmP9VLF#U|umT;SuryeTGj&s?!O$6|XzT?_^z|TXBUUx1(ccgB-;$kH!b?E%a>GvqNRTY<R|m7_P35os$|2RuckaX<&yCO^4l4eK;0rY<B0HGGG^2=75g!FFF<<i%x<eC>iB1GX<WXM1&y?g@$yL{m+CIoXiAd_~~A(_wXNkS|YP->fENkL<oS0C4xZh5)ojPO9bU-mWTnc|Ib^KwXr<sef|BqgpYFGkw~byiG1x6nINyVM5woxCBgVR0zB#Zi-KbJ5;2e$St4GG+_?k*X-g#AI^5GQ+gtw<sZw7@YlQCjFA;_oNlh;Nm;p8U+GG0EDE5h{OT<f<ar8<}7`q|~dRU(}<2IGIBxZd!IG*(|LC?eJuJroL)vbLat0jT@lh$7h9uD@_VK0FIP-TfEbUV^y70-vd>e%D=mk5CvwL};c2$qQ8>2_wjPoZ4h%C8M#eY7tQwJ-7KE`O8|mPi6<_4Kv(HK`>s;fIv)hZr7J6~L61h&B=P%muLqAL=j$X`tw#34^zd2K{b{3HQ0Hj~$=T5wa)kyF>&!ep8TrSt50x$9Bl((1CS5i%%=a`b)xK=#5i2uEi(Fa4r!l58@q!;c0e^zg(NyZq>RWTWOyCxDtV9(p-~%8sDEAD1X?&SHz479c9$Fq_4dN10J+Q1Rjr2Jc>c@)K$ptbfe|&OCX_1=73l+Cybd%@SrBvZ3@7OiAh5m^nvLhxAM>LYyWu)zeq!76M`O*IYyfRSOT+O?;Hej6-$Iczv&V|nXe_H$f{?@#_n{z_r-JkKwrOJLR$if+{K&&+<RCe69DWvsmF=4C1SD15Re#KB3crcBVcfAUK(aa!EuS8s@M`~s$!d+P8vWbG~2d}C2+mK$L?(uIf-k+e)QV$u(KLM1%@SJe^Q(H?FJ?}qQH{^MU_bjY@P=+6P}1xP9_kGCqq~;xk0l}4sVGlo~<E#KG~r`a+Hli8(2@qzCd8kYf9OW^bSIJ5Qa<cXxQ1qj)onr)Mc@AI9?VO-+hcrBB1AzCU%{ep#zj<mPnnI*`cYeifxafkl#)fgd3)!Ui&42jaD5`H%f4RT(QDL?X*!wr=xtEh?q_z1mCAIBFWPPAQV&36u3kT<*KH6o&DL4)&lfkEs?H2aRV+#owZA3LThqtw$6Z$Ocy}b=S#rq?Qw(H&K;VH2Mi3}f!V!7F9p+bx-n6jV+Nrd>#VC{cMO64`X>X6k4+A%$>7dh&#owTrxKtLwL}sg)NCBW-omCu@mNz*x#Dw_6P6Zc3jB<MI~z;HU~oxC*enBDesf>!*<yMNV&z1rYfjwon+Bl+Mgt-*8iEcO4Mkj?M(1i{PuH6RcJ$I@X&*mIZ$V7I+jpP0B+zH(L|aIcpyDROB~QU0QefCL4Y@a(O|okfPnv-j1nOZ#s6@U*8ho%lAQ9nF7zJ*6sn~~BDED;rHzb-rlurf@Dme-Y1n-J1kwlT5cwmvM+TV16cuSzB+9krE!EuR5lTY6J!;0b@RhPvM+97t(4zol8v29|Q>LMA?ze!VTyrZCw+9kr!OTJ8uEB_`6eY(`?gENLhTM}5sB}sTpwKm$j%{u!W1Sz3hGAW}p6KNB~j;W|3KtyVFYV~;9mWds$M7_x?PCg>pr>n>YA~$5%XboMvSt1M_oC&mRgUttIu&aa2q2sZDreN6F`Y2D((SOIq&QxA5W(71RXG9Ch1wnA~rqFoo*6RxSQLTGpEJq};-m<y?yywvpvAy7Eto9gF5QGrHP;DX5QeG;?g(D5CxA@Gp$$%d#Xo=W98!r)~qQil@=EO5d&OTv6*?8ImR9+%hHKb8uwvgCh)1Vck*R1g@IH0%AL=c^uO4#!m9uuJu9xyG#10q^(iP)ok9*XBu`+_q@W3qQ9s8xB1B#Ik6(f!6HGRp&Ywr4h3nX^G%J#3pkPXU`ewK4q1TkwPomnTXd@C^JN*z(CVP{#*Te<gPbz$`5jBd5bJ3zLG5b5jpH*?K0JmSJ+JH)>$loeRt*G7nUvXP)o?F@YLAGsDAyc^GqJwr3M-9`;m1m>nT!W>{+E%fsSBWauN(6~uN};FpK}j)MyvJ`#lNO+b#8Uk+9UTNZ`BO$_047Dq6KUlvxMV}WfltPpi^QxAJ+8Y?U`vLeMQ#IoEqBp#DiLvV@Jq4G6Q5|H1HM6Nml*alc~TMgU0gR5a|44s<^KttR-qIkL**$jFM^EPY4FaP>TgPbF3*@!%eImYdX>=U}`mw<8QEu29*nY-&@#~@t+xnjrh%N>gZO7pU#JI>G3w}9v|reGWdAyqJhq9Nr5-Kl7E24jE6%B<&{`!kNJl^jjl;JCM!gMD<F@0sL)kuoP1p|U=E9Mem}s!Bemz_^hLH#rS@rluNpzY{3O!I=oh$XgGS$^%MA!?sCEcR}oah>>%W0aJ-G6tRBk>S5dBWH?ftjOgnVGO%tZ#9cCHxy?Je$b?Z$=2sM(KSdgcuopZP!D8vIhutMRa$oWivEsDLtaCEksR(9a(pAIGieDb4MUhH~0jm^}538PkhC(5&Px4k;Sqs}`NO>ukRK8lep2VYff_VdB7{a8x9Ck$ri!egy@{@zbmxQ7$lrYR3$jicJRWlHdx|+FtPFD;&v+Zt&9cLgDPDQ9;z<WV&F|3OHA_BGF5wUZ^#Gos#`x1|3cQovHuI6so$zq7&81~)`lhQ>hHN~SHpcLT)(XTS&{FB;<D#%cXT5Sh+5HC0R_%035?k*h-JIk?w=GyVFV{S&E^eP<<yCQ;8KVn+z{wU?;VI}Gn!_L<Jj9@<{HZT@Tlo6AciPhSFnb^mIwpJP<c0a?2{)jiKJz&Qbo-l7JHv52iquD!^hKU`Y;x$`1N{NOooE;2M8<%L(h{vo0HXTR^jD(?Zf5IAgcSJti!Tk9h)TAu&g%9SB0F|&+MzT&M9^Gecvu7sBkO!ooER&=n$eXaeOVXj?x5cpYp+wfCWAqhK9TTOHbPy<->Q55(VJF&_!XBOIfT;r=27-^4x57>@l#V)nZ9uCdYop&fn**4d&WVnn&M(L{|3phVp~31)z<y`U9%QFeQMpaa*XvK3QM<=E_7T>fWw4JVh|a7H?W~vTo44mPG|{<Hf;2&2{xypLs6VS<--9SJ0&1F%Hr~>o6zrqrM5$KnF#$@blb3)|f!&p1o=ce;9JD9P4AsU0PdO`~Xv#2Spj?NptCYY?#tv2nDmtjl{vxWr?CZ-O_hg09`Fh#cEaH-iB0#1c`MRSZOrR1dQ>lXbV^jya-l_W0KKpYfN^J&2l@VB0#rFEiync{CRamIepVaI7LxK8mRADwoZ6Fe%Het{(0_i7x*;hqpv?X6>5dE_Z{bP@Ji?o^{2WhZBMlH&m-JM9n?Qv|fK5Ku!C+h^gl7A?6{s{Um&`@6LRp!mr4ds8;9i<f1n*!riT%vNR#>cg#pBogj#GuDg0$x+lU_=x(YQ(kG7BGLPVOYe=<y@6S>Hh9a67*JRVw3&JyuQFvHc0j;Eo*{s#=vg&C0><6(m>Wi6TBIr*>!q9d3XK!7RLt%u>s49HXuK+QI`?6Be&7O*lqGQ>Ta$(S;)HVvaT;5JoX@7ZOyhs8&UTG9SP`P?WJ7>*ReSP4Yf(=;MrtwtZYi7_s2+VHd@^lZ_`jh&^q3=0h>*j*|L{-%>ru~4RpW3XoB>Zy}awAUo4)95LZ@m=OM@&*~_}BD^vEeu5V9cUDfMY2|;TmNDPbzG&P>}5#QU()Wp|qBwfvG$9ns!dHo2U##$Qv*4oRss^Y1!qy6Uq{@FUdb(Z_`ty$p~lL-<yCXHIp2^Ox|?j1!Czc+zcz{F~pEKjfyZz6E)K>`tClLV;-yOb+FfaRHr))sr>iBhTjE5RsVVJbabFjWL9t-JblKXVit_V;Hb^Et#Ch}xMZ$~Tz}K$*>&lFSvMQYUBsjC*8iCJ6JJ!7d{+t4G<pV~#?oMRU(^FE`;i_fg}{<M|w=2&{Pm@k$)E#&7R=PK66fBiJY1`WgM)>f?7;?1(U%2Ly$Vp;k0r_AI)rb)Bn67F6So^A*4f>~gKfBSn)S*q%HLq!8?Ktte9!7TsNhZ71EGuf5)c4F#4>g8LlvOSd8_drTa=Qc%j#5?!{puHWzgm~=dqsE@``Z6-+A`xwCG_-J_S9fT?ex)F?zAFklX7?QllkUBZ@*rN}vzOuK^hoOSD53hU7Y)eEv58m@g&_Cf55?P<HU*0Dn@92{OXz(dbAl@sU$P@UqIvUJ95eGK!v*rd@Yl+~R(<u4!L${n@e!z)PTZ&U1xsDBVb*?*zmV@kjKu#Ym8uQPRs#D_yu1W$_1e_8iqMU2BZUf+^UF%o?6B>hS2D&{BOKC*Mnz?iV_aoLvMrQ}vA^&1qKai_4^r|vITHIxV<bn(JSsRzyN{`zWfu=5sTD|-dt=f>q`viqwn)MZWADrNljfL%bXw3%BVm@D-|B)DQt=Yf^8?FotSp^_pSB0KT4csU2rls&4IgL7PcO<gUFS-o`XWS+@Qbuj!$gL^6oLOAB`s=tz_@0EK+g5(*)$c*Bja_>6<<|F+vbwYQXw<S?%(0)#u0Cx21P`C6_~Vyd)l+qbf&J2}lBT<(o-Fqd67bUvxg&phsa2F_=m~DWdkj$4@n~@B*_xm-;GU^oX7#ao$Jd4Q89#rCRg_4!VOD4V!x0LNj<0U5ei5sm)|>+*Y(48lz|ngWoJiJMJsSDE$Mw^>r`DYI>~EGS>?_bDShh819jUHFcftFKyYYF6>qq>L0q+*XAe5^M*gqH@t$H>mhW5cbhBal%=h(!hRpkc92ylMm(yHIy>Z<kg8IAep)q{9xRbc@!sno|l6EPVfk(dGwR!prA#Zl`eRkhBhqf}l@*z6j!xeK$ieL(|UQ3--cfFJ^qF{lOtJ3<Eux}g`OATt?cAlw8wV7x&A$H*Qg29M?{+8%>z=XzAe1l(_2TJ?dQc4j45&}vzm^o;F6y5#kliPtlE;6`wOf_S{IXH-ZqWFTxEms^#I=a8oJ;yCE9#=#$CG^9oX=SYYmr`P<V*Oh=xSRtqhD?Z=On16O~gbrkBExUTSb2LS)b2yb7pT>I-0U<b4kt5N%rRBI^g+4$7aih>CwQ?eX4S?k060DLOhY?~wVO*ll>jz9g{yPjdWQG}BjsyM^76AT6$FjDB*L>g{zV5cxCP?7~@|xj|xgJ#Vd0p%0@pFZfMT{UE^|gfuDwW0MS@qt;06mfsvw=1{Y6bOax(Kc_pF7NlKmZkrXmoT$>rB-VYm5jOalRS`f0`O@7>rA`Y8%Kt(-M$pMG87tSyuy&mq<e<lAx_EGEnHutfA2}(S((&Y&q7Wj~rB!^-cJ*;aJ;1a=^Z}914(64g*Q$91e;-Iik}0-ntwKz+%n;_+rip(y%!Z3dp8caIbZuSd~+I{GAh(uXENAU<clubusWxBLN-BORhe^^))f75x_C_?sC?S%t_7R-#bSFVt0~)B3+UJaFz|Zx{C=Ag_2O5N`^gG$rDtRv#wd~@Sn`B!%c4bEbc@V^_3(WGIn;<7Mm}(`bhTLmS_(DmXJ|y-TuhStg0MM3QAn3!0zi5K`F%)khMxdQ7Xj|6-fzXLXt>Ap-0t{uO8x@O6b&GURCD$^QBc@ZNDmr%6Vzk*Zyx;rm!U_6)IJ0r+S>-WmR9x-GS_7Rb{9%0zt?a6wcF73LxKmSW->{l~>ceufU}R+7?R$mVES?)AR|jbUGn6oNrHHJ&2&N?a)je^{b>ia%4Y?R-bOlvk}?HGGMkUFQ59bkUf*jram-sAVx6$3QDSEDDpzlG+G|5eRc%?#8eT$&@P#ZU*;~RkKz;Ms`BMhTqrdOwG?L7DU}<SQ&6oh>)Ti7;HAM}m-Q8avw0a14!w1&A~HW!tJdEwiH7?>Cu@mN?~9g1T?RN`ksd)ViP{A`e0xb)-yE|fya!B6B7i?zf6pFLKXty&+?T-YS;&$gw3gX4u0E;RB_efF{Cq}{flLEQnI+;)9`YYIvP7aU6fKGD0r68B?Q^qBWP*pdmk9NRf+fLy9cW2N-z&Hz6qz$w-FD)S&Zuc!Ir9LwT9-uDP-GiCk3J{9Bmf9l63ulBezeYv+zbmiGjGAl3QoO548-px;%IX@T7tL)>NjtWBa+ej|0OW{#O{(nNvb8%#j1V0McRbP5>TWLu|DoD36T6>5`nyofEwV_u~4;o9fznj`ac2<Pu=QJXr?MX#f+f}VtWkj?DWi}#{W;^Z%M$zuVr_|QHTuyW|s)c{4bF<L%sf9($wp54K%GS2}P023{TYIAkin~mITQCJ#8_-U3#5SIS0lGhC1iI^>Il^-)XQUw9Wl20qcucOTxiryd@G4Kz22%ZHF<ClZiS^ov#GAZn-2FD5NV<-LGh$`&kkOfcp}0<aJ>$;1U77fKU5XPj{Bc#3F<|Q8L22SsUwt5UG9N+mbK@WzzeC%aW))z!5-{Y7S(ob4+0Fb5Kpp|4AV&3GQLa93W4%Bow?^vqTIa-T6|g-w4w^ABg};pCbVZnR9>~!xEVwA+kiM&oeCvE<xr2%yUYghFlT~E~hRL10i5eaNl}K1PC;9BDXqe^m*AOF_ASlZ@6!_I;lng_bUQ;$|Q9BCxJGVB~jL>k0DZ^ZJ|UaR9~#x-{1)j?qyT)yCdg-;mO>$4=o7+3}e20N<Tg$kA#72$z(_FfdfIz$#oO(k&xP>9!nR*QKwKJ+NU7eKf4mu9Yo=B#n6Fj2-S|bQ))Xs`!9(sL<c~_Q)k~evLvX<{A?4^zU;pQq;DZz64;-rA(S$IK49^crJ9Q1|I`GTj%iS>R_CeDJdDux8Z_3IQ~l1UYd(f(+E>e#1gNPj39p^V9R#JRmq?kVI^2G~q)IBC`9ylhq-r{7rdkpL(hw7ss+NSosXClTpi|Rus$<MCP`Yf!tWyX(6up(I-vsaiz^FmhU2sTWn_Cjy6|3!8!%&zA2Zf4RSE}~jMIuAYfp#V*aw16xmL>P;(wPuZHC=yq449{yiSns41EjHMCa9=AmsQ$6mL)O!_)c|^>K%f?xuk*2lt$@PkCoB@{S-^0VW}JI(f~A-MuHBs=Xy^IL}@nEfBOfTsG34AqpIwXrZhlwPXy$tXx8_D?_wQQ4`_l5>q`RFsEDAQ(nQ5FBI^4`mINcuO-+sJ&J{ctyd(@%?Wkr{XX{=@^&5rQo<%mh|AZLg0XztCO;XRfWJ1KA>c+pL4hFrHOQP=lxofc!{Q2B#K{1LrK%66nhAN;$O%0<uN`Dzuy>rcT6?YJXr>MD5wWB3<jOu&^>Qn-_nC`KPoh+l5Q58)rOJbr!sf0XP-Su}hrckE}kG>j6Kao1Z^`y~P<VYLfzgJr~{oS>66{@{;M#u^c+Nr_FZwgLSbB*da%a=`2<vuJx@{z0y{ClR35GwgZrDbvgK$w~e)t!g{21<_lD9GzRiswP4CDFfBvjhUE-4o>wC`QACj|j|AQ-IrPOM(Hfrw{}mLXjJKsg#uAG8}oTfQ3JbM1$l&&OMuQ>XD!-=)PY8{b`hfVs0wP>romggj3cI;J~FstrV0^E5$!L=kX)-&)*3O9utI19t28@+&=&JoCp5NroLPi4+VIdf6}RsG&YY~sXIZzBcMcraLBU;soTx)L{&eY;3CD6VDKQy5|N%7@>I0DkWTRgxSwYL=g+eVRQtd%4YV%=b=vg%-LEi!<V?t;G9i!3R0IDzMx9DIN~M5bNdRQH>GwM)w;@wI8^eqUDs!#V9a0&RDQ0M=472`RKTvTEgag#Um>WwypOJx9BW26~K~yX?^uJ$WbRI<dizQ0}QB)Qy74i<^ZI+aqhQGVmM8{H_JCtYZbfRh-D<MNvR7znL0CiavWfxctMKX0jA6Wxr<5>}&VjCbI$2I{>VS`G23Dqn>h@q+<H!Xj63_}OT94QQ4qgoQ+jjHDjoVsBDd<Ay=u@iE-O>OD`3|O0DSAY%N(6+cEr8~IlyCk4W16t`@nW}<zpiBw}qoqWx(X*77N>LRG!w{$o)R)Q;=tx2$#W_H(g%iU5r1px{5@3YKvaU-X3(KFFe}2uQ0aM#N5>-(7Wl|-G$!u@wJ<n40+aja6=r<n$0tZe8c$j)gpx#Lt3Oe6qSYHg1mqk_hRbCdw<x&!Co4hQl{L0IsW^wRk5<prq85|aw&B5Oao9M~UC1T*j$#eh;<mFLCAeEUQLnefnsn7&RK?r60giz~*5K5s6Dd>BVmqir-SjYi15kkIH2xL&?<xsQ8!3j@vteF5L5Jr_#d7q--YNGJYlrXCC2|Hj6!U24`FpBsh4Tw4iTP8ZTFNLb2u}J^BYD+-+5GSI@A2;==4YTB}M-^#4kwAV>5}9$+;CCj9ZcK?1NYIO-<cTQCoJ!l@w|&RUo@O=ECdqf^5#Yj%Xo3QbSbgx1|1s!FjX>LtT=L}c3joPd<0B%79E(9aHA8{Y9F75lzM=MBdHGZ2U%c*^Jo@k&n4XPkewpe{LwZIbFMFz7!^@sZBXk&^>c+lzo(U=<5(8Wwmw>J`5FwR7*(izf`su-?p~yFcArBOe&QsJFYA03%vM0$VJe`Sj?@#Q#O+xmOjBsbepN`g3(Xmd{fGX)A(=3-h`9}=nEs=HA?Xf}~12xp^m>{941GS*(VBmptG&PgQ^XpK6<#iY+GHQKQ>Xirpwe+P>75&kP8qjr4U}QQ8Y3x!c&*Dt!M3;KCd3<7@T&)XHS059+JhvnQz-XP>!)Wcg`ku3|1RhCS5`<a^lu-YvLJFk>GJHy9jJifh4p^8{^k_^Q26fiT16deDl`?_*p4w5UGY<j4K^dNWHW0Fv9V9-qp-|^MaUvrWs8jSx`jQ}kl2s5bRY6%(eK}O=#ROeERdKi$M50ts|Cu&bsB1zx!R<y>l;l%okO<PQsqaixK~_T5K!HH(U-`Wzq|luzD($JEMoH~n^4niAf)c0(ttJ2|sVV3!QbVa|HAnVgh??ppQ6J%Uqb-F^ecG|8t#xni@|+XnsQXl1z-2CNc+^puTTjK*VT|&!@%&68L7)s&FVXH5zinWFl6)H6|2EB%5Ol#xqqfomg2Ir#{%Yr&A4fmx9_g#IL>@Ososm2$$vVS-%p(c;vE*858`Hf!sv?D&iacFYhB^k24$?uI0mncI6dMD4#YU~uok6jIde3YyNcGqNzdExdnuPTIjKPUGaF;ev_nD3C3v)KnjZ>Smrwumiw$n4u=5o}Y=KSeQ=Wmo_QvoorX?@<xruXjFCQ9&IQ-<1C){F)Swniv{Vnl%L8F5cdjOufLKk}ob|I>GD*I+e?x?gd)*Z}y&E{XErZ@^GKGvxs+kkM6hV}cS0Mj4~$5RDb&&5hBmNaNB-&!#Xc=o@EWOf?zk#7gZ*cj?Ap8=nbEMVp|GGZPe?tZCzW2SMi^_VJk{3M@KMm$&Ie?pmQzl|f#^6m{{MDx3%E;x%<B5uiwEO&8x;E$remo51PACO&JL_%@-m=8=i>5tQOHGlXu;k&7}^GqH|8?WOntfurl;+aG!9Q+XEZD^d3&f@XpG&CF43a)e2(yAZX9sGsZJj`WhJFMV$}851xP3xaVt7#+4iFy8{*+p#df^DG=4#|Tgzut51eYdZLjTgZtHLK5hzwPZlQEJ2Hez09f7IlKvF=}-dvT<4NNwGtnQ<NKI^YxtlZg!w2z{@E07o|U*`D7d2N!vMtc;Rp~o<V`*Rgz`B+o3dZt^nq)c_892>BcBXhtxph|d_t8@gh_p(yuMHHyN636>q6?|Dji;f^1A$VclI);vR>!JAc5!&bvjBXR4Z{pJtt16!Q2V-L^>rp3tY#i;~HM-M8BJGwgE9}F!z@_eGKtTw5dKFox>L;oZ)pHztpKjKh9;g^z1Brq|c>R>0a!mPPGXN%IkSue7eTb^zWS`DJWs$VkmIb%<1S$q7Q*CsO7{Jb(y-(9)`IRI4OUzrukuC3LvSgf`q>-!uYQ6&^N5<ETzss?e{tSXZPM0p!A>Zd!L8NnjX__V1}A_+{#w7v$LikdKn_3-u));gOA+M8*={hVZN_>ZSM84KS#mQ!`$u!R_hG1F5Xq0_P%x9dwUk3=;U7ZnBLgs=z^~Q#hm8HAb~1*4hv5nggjWF)(8)%(o3Dbpv%}21{_d0;Ike9r;&irdJeF);J2f~yVku8DH2?H^ehkk`FoRhzTZHXNIg-HnKuQeI|kxSF-EVh7^uZ88rrlsEe23%41q(S*Z5*I=xLwF7^u-CM)c+tBZ2kAtbg8Qd+V69ha52pybKc)U3iQspwl{FxhW>9HOKVE(znJYO6JEzwdQDe({^z|=rUPA-V_9CctPNnPLTG{CaCtJcc~7i8K8qCn%{~#M#|`RUhq$h|8aF8E^Ydt|02(>G_RW<g+SgE&Epa0<g)=b8+FBLN7>*2jc&m;xj!EfR2B`HppQC)fJ+ZSkTgVqw-p8cN6>Zr5C^Z9g#=s)dMi2<Wb#8NSgI0o9Ey@9p#s!(X;YaZ4>f>|;-yVhb~|9ZUTA`g9$^DiisI6yGKUogRf?XB2oq2?4MQotFzbm|z+zKe*7V`OHs}ol)sg5p)800pAZHveXR7SW8Cj1b6Lm#i!ws;Aa8S07pT(w6FyIN`W{ehEN1(@dBB%~rXAG2bjzFoWXlhOOD|J*lVLBWT?g$u`A`zfS)KV6STFN4SlF*OjQY7jri@Ml!@n)jtlDM>~ysH%Hperac5bn*o#=Pf|1ND?;)2DYO2src{$WU{DlwQJeQw{_EJck1t%Ml=I<Up_OXZK*lR!8$-#F@t>P9O8wh%_hir8(=2-eX}o!H>c(2}4Ss4)WtUQJFdaDmTsF?luY4nG<&EB~kxSLlQ!yNvL$6q^LDIFLf#lJ^4~6D=<M)fQ)ax)Tx>&g9KFaM4Qhbm!HfaFeD2oswboNl4L_G@g8C&2mD5&%eMJen|=dudXB72z*SQS@T$D5>BBU3r8-5x{g4!tBTYe-`)vC3jzGZ0COXrK($0B#(^1F#aFmL&da0<XEERR=W<#C!QBVUUF&pM|&UDS34vI?67IkKG&FLt^&{c;t<VOu%#7zUtJ`LT+NkfO;9NDl1ngwdnOGlVA9ePnm{m$9cn$A*ClU`olRCc&z_fo|9h=YFC^nha|Qb#lm&)Y4;1d9y_NSlWuu7p4d1()bDph^R^=4M2Hq9Xy3$V82g*_4{jT9c4hXr9^ICDpvNsZ94~)`96bj<i1inOQ@pIUc`|QI|W9CvIrdKT3Z{Q`?lkBxd`Z(h{J4SC$0#yRsy}1>7Y8SkscQzMF7KIGkzUs<Q+@vFRb1ZKiff%-#tu32H;a65!qfmxO?AYg+Wqrp*35xFz7=Eu|&Wzh<xmKxBG2d3%<YM4iNc5_(I58gw=J5#hdhcS#7+7-^rSSQ6I9geB1!>3CGuwCEl4phWX%di0Lk8`5;gAYdO1;M0<5%l7N<Ag8w^tk1_S32(vC5&-4UC6UO1$w2N&COFgReZ3{Y<w@Y!5`wn1Y0x`fQout7OTychq9uUb=wCOR=I<eNl<l1lmRu4O=}Q56S`v(obn`UCIk2k8KPh-;Yf1Ep{3Xy7M)t;D!t{X`o*Ru%R<SG&Zy#tA`HlKzi07925&*YGmPFn9^mJ`63GH*2OTy}_yT|1>tZ7@Al}38DzAiOgP*OjxckJ9}Pq&u@q4Vh6J+`p^E?}Xq00cTjPIaN_jPTPu=f{>gcn;>tSc7m86S(pKmCz-@$WdRu<fW$$ML6=WO4FD73h$FfOCk_X{8?)HDDOQ<{x*rWB%mK{IMbO(a{uVel8~O=EeS=(?JYIwhm~z?)Uc-G;}Rj#tgnXeC_BgKs~$^Yg2*!m_0yOGg{C=BXqrRe7=YL>i9Y4L1fnmeFNv}udUkfsK83#o`UbrvP}f7RHYT@DY)<J@14}}IhnlnR$z1{<EX^6n{JJOg9v3ztWts#YFIW=PYm2xq{wxUru6C04t>;U^+ET<4XevyPOPGY9)BNi(i22(&P4p`Hl9<RTFacOInSqm>+&oxxP68|PJ$0Gs4gs3+mjrlHZ%O=_!2dbXvsxKVwRAflLB}4|nU(|!_*0tDwD(rPEu19*+R2uL_vQ8_@$>Qg`RhXe5=r2UyZ*erTAfe=g(hWykfsVUpr*`p$1soxnO%q8HV+LH05y?!Fi^W&60=3<(+Iga<35!#n>y36FH;bCE@!HG>z0J~jA&Y*jc*D~XG<J~dY?#uYQJ&z%o)L&=J%WqYH{t`g_eZ%`|b|1w9{*aZn!!<(HJohY>C-7t}F@AH@+l#7CNKP<jqh7TEfste7p0!I!8@+-T{Gcl}QRfQD0};3(bk*NjCO*f@8v<lmd?PTVj&$I_Q!xwC#xsBC}yj$GDo8$hWJffpViXK$xdNH8*^=g$8&@Ye@*425_MS*%FNgImy;W^sb%;O5{<^mbOcvy25n#WD?ML)0MXse%Vs#GH5D58g$vxS6SvR#Q`{@8KCKIiJ+)NGysky0%3^=Wl4w_PDAQL{g6LuDANW4gu3#U1ZZ_!672)8>+hge|8yXhiI{C3+&4cg0YY0}Kc9E)oCSRZL3?6UvLkjt*@*+3CQgvJBpEeHJ@(a<n2rD;0j?wL-q)tWbgWDI8+S{<`s)Bo0+j4XP|1#@?;3sDTibiyOJcT=hLjM0C1p^LBNdRUAy{7`HIP^!_4+4`Hnk*3l}M3F4P0Sd5)<GvG6Hy)YOZuYA^=E|fxgitVaUzamzl2bF_38{OCaC_P&V?3+^nNWU)Pt;fo`rc?R|d>O!XwJFF}!`6Yqf>q@&gxYF-`RwGfI405Juj6CDheQeR~{)RRK{@Ed*e*b)F0dWt}9JQ6vTZ1Q(|r8$x7N<=&<(Q*Iz^uzqnMrspq9(5#0+f#xn2z5W%w2x|LAAGsvN%IkyXap*=<vX3guoaJofF1xIpq|X5zX-Yn5UA%7pjUxMowz&moJgN_QR(QSxxBQUCxe<2Pn26`^i<xG06&4VBpgIM?(Qx<M}Q~l8Dkp2{Ahx#FB5|FObpHu6UdA(q4p7`t~|fDp1@7eB@u8fS=Xz#cjLck(nn(=ca3<Dt7axNR|Z!>mjvn@V^;73CQHOXm5kZZiQ<40Nf0wwkflofOy-%!rAl9U!e<tOpgoJAS=6{`^F|h{N#8!U;vmV#62M&x%8sy(=H-zmkh7wKGAq@U+I=jwW}nB>|6)s@6G3!|8@hBQ9H1lC=n)ueB7GPF_iQNM*B6+M2_mk<CjIr?C4r*7^*<lH1RR}LLs@3FM9$U+ItUKl$XpWC;%)}5c<d-S!Y@_g-v+$X_S{sJ_TRl8Vke0CEi2W5#}h%t7)a%TS{^t+!ij^C05s6Q9?JXhkzxJ&UrWM4K9i$=hHMFd><H&Tup}qQeQ**2X~`%l#ffUk-2BSzL7e%6(Erumapnp^eg+CW4QG0^E(2T%TN0pWR7Mk*cn^VEouna6d-r81$RbPk>cis%WjK`RU&dVm3Hl?YW=)4F$ebV*Ba;B(mWdM0GLbKpzv@dLrY_UKd6D|(nf6(b32uTe36v=jf;uJCge%<@rhT5I`F*CNvW7dIt?S6s`HBJTEH8JeVv96AdE2WCAy6vK6{f9IM)&N5!DY}T0g|J_(jfH5zp&O4cPR!M8-zh?j&MMIHII{Qz$r2SArYCNzAZu^7a;=O>04ydZ}KIg0PKkv{0Qj86+xZ2BH5=|MF%<wI*~(G>H)ZPMW$^%D#$n#v_6Q!Q?IByM>Ohf&HWYBjVCrhwpVPzF%XIpW6)JB2E)@!oIXJKu^&T)J243M#o)=e1oav5_Kmmo_X(06;sgw+dHUf*ILMyFwPy?Bs0~*fp;B>>FA+yvRAQi&#6+Moz{6QfqQ+1M?FINzktLxhp(26OB?8%#1Zo_W0JTubf!s?@)W$D~?uW_CpNhPnK=vf5pmABcH}j5!qrFX(CzS*b5k9XH^`u0{1h%dNVJL0r(_XC(fnQNj4^mWg(%~S2=|G-T2kLv$InbL3`lIah@k<#?f&eJciEfAK1TVZT32hj~-k6C3lune;*S9~9ibhp!2-KM>4Zs(aCYlG;#nH)I0+CWB6rz+AN`NQHmPE~;rfmuEPVSP(A6oxNG%BNvsWPhHD-(cp$_&7VGCchqg*rN80~r`?eyAz-OuI|WTdNEn$bQ~e12**5p5*&spCy6H&MMqD^{Ak>G8HX%?%0QbrU(^;__d)>S0a%FywapP!BuM2x|(+!sv=UV^`ql^=T*Vq>#K~%GP-y3a?i4TLu>s#`@h=&TUwT!u1hd<iBXNAgn*OS-DTRdKx|aak<}Q;=&OO>2!Gx=>`|IJ_&vrY0Tt@dG+k<{`P^#J?x$_HX*mq_?x29H9>~N!QQ)ed8$ZF|rA}O_wCT}1;HWM;9c{$VM*NQonua3a8=&h^o9D*2okQNF(bVzwJaYi5sfjWsnh@kCc+Pc6p!+wPiZ1r)&u8=I*C37}U(>3$x8<)_^X6~nWTWZZ*`Kk7I_;QauwBmvgXm~O`bGpBO~rYmNgD>pm5u(#?3V!0sSVU#W^;h*r_I@)C!44_*CxuD*hD1~n+i<b+IvlV-zLINRu`Vm%Vsvw**8MZycvO4a@RlL2XNLu+`BO&0bbvzL8Gxh0`H1h!<+6P(FYfU7n_Vx0@}DX)8jq5n@w9E%}WsHxV)^%DobVsw*ZYnAGxuk7k8|_+H@eJF@pFe18H3*lt33p3|?+BLHSY>bk)HG{0Q@saG>><HPs(M857f5rTa|k1Xm4B!Oam<^jwLlAeE~k<ACjYrd}e#CueJlE{~X^z6i4cNV%bQJu~#eqZ!w*b8`-}`ikfY=cXtHX@>eD%uoxYS${{=tU*y{2p`DHHTRp2$96(lOhY%E%wf!LULC*lu+j*3TE|}g^n3fq++(1ezBz`bfgXRdm{5YyxeZ2*>lT3KTcD)A1*+oPUli)g%A)IrJl{!BLbv)^h@&z{qO57&+b8DbP(|B=CCchqf;zsX0TOSCx_2xCf#^gfQhx~)SJiPJ{ry`X1pW?wSZGP0wmom^PTRToKql(LK{)r(<W0_35|DVG12iJ}L`8h>`u29@@QL1j^5)9ZcIrN%Ql|m`<MlgwCll{8Kmpxnf-Vn7SDu{EsrTkbZjTibg9GOzsGr+O`xxg0RZ0#}lD}mqDHVn&sq;V@d-;<VF3Ov#)BQ?+FV9)P>p9NoEq7-_PL3U}qz6&n{KoBBsX%=S3isah>>Udfl+^d8;<Q!X{8-absDPgmToUNPB9{O#-38rza78yCTqinA&26U>-XME&Mb&0k1@PD#_H?wa4(EaRsB8An#%-WSdfZUEt{b{6?#56;z-<vX?OSdAudcn1?4CpR<W1M!9V0<8#hb3ZCz0SkpVIqSt5d@_ANz4szIPYE;2pK?xud5E{jb{7{L)4sh~x?M73dWxkBRdB-f^d`k6?hoJp}yvfk)H4b4;uuPscfggN=#kyWu?0!LNDKSx8QFufY@D8ux@yKcSVTqF|kd+EBcHC*r6hK%U97dyM#5ZHxg1i7^4ZiGj+_G0@46(G*GVXKJPHa2x~S(iq;;;uxsx93%T%J~0Q#00k;O#RMJD0q;D;>|0D?qQ~FjB~Day0>pGsH;9Rf&e0T}_Kr~BPae%LHti&N5bA-7zp73jsjwh)?;!~GKt}VP-rkk&7n}Bhr2u>-Ufxvp`3D0o8cny}SxWa%#NPX+Tkoue(Yueq+K1U-1Ccto`DL*?2=b)S)t&Zd1OfdJ)TbZtzEd>Z>1a#@0D=%ybB^Z1)3J4NWFOLWK%S#7&6_4?y$XKfdPz(`jzS3_!l-}i_S79iyYO^AgMM=))X;tEQ1IARfUdnz^!pfL12q&xckSs&K%E*REEa|;dQtPIV}~z)lA^CJ3`KnVQPBJAEt(y#4Lr9M4xWb!_~T9CsE!|w8hFCd&kcp6Mzd&o_l}2g&}kMgd#VGF_syjNAB>2YD0T#T$|$<R(-jf4aiK`*VG!Fk&yk-96ak$_B!Xja|K2RhhLPYU-DoOL$4;Bl)3MW+K$XZc72KDNG?WF3bo4+<WS|Sxk%{_N@~;Ba?|N;H392$V!0)u?U;TR@Q_YS<Z8OU5@{?={)LWi^-G2IT?KuK<3g$rR{hZ)dem1X1903AnGn*RJS;|BWXF0WJggFhQiF3lcxj9j#H2<nWeeDZ@Ul~mFG<g!js7c_Vs3ZaTf_$kI6}iMDL-Rzx-j)>bU=#AF$>@4+GJ^fdh@WOzf;zdxWLVWp*1oeT+0cP(UfjEj4b<zNjINSq_dNMg2&_Y;H0kJLsLkUM4+_cd=F^$SRqF4L?0}kr-s4Ddbk{fqJQ0_YeR4ImDVl8Lm-QRvN2T5C5qla6L4Vd)pYAl!WwLDQP<I~SC7x7pVJ8)ye5-^0aWDqbd}&RO#HJqE+<?0C5I9p%g-SEHZ7+?g49o%vz<BTIc1b!)OQm<uMQ!77RM<=SdPKq=zH|dbJRL?Tk2(w<6jc+c<6UCFy?v?F$KzKAzXs|&$w2Hhe-xsA^6aDcK%M)EUMtT)Z*pV=co{U|J*dn98agsT18*jQp6P6^K^^-@fzyCF>FkKqaUhA<X(lS`XLk|m9}M&S5^mpL@V^0?(23;'
_HIST_VALID_B85 = 'c-n>3!EVAZ3<lu)q`e86lGusur$H5Pm^SD{Vb4D&tOntda;Pfu$xeT#JzbC6>(73;9H!0pI8B@5Fnt~N+hMnxZ+GK(-f!FDZaiJi!*u!lJ+wdG-*O82+k}roQLn;!39;UW3>b<E=m?;!0%uW*6R{)}*+h0CF{&sX5vA536ZkBWL{h=C$lOGHBm}jMbV*RsHgY4`lM;0+C3Xpc63aCwPcVrRSR^+AQU&&kKw%}8EV3&Svrea2vM5fZs1q)hEZ!a`-9;3PD!3!KZo_iL^4z2WvQ!a$57HiFp`wc;QmbK^Vsteq=?Nf<6vT;)I$g3z{Wfwv%w&NAc@b*~3l!K1WF0<Ppio|Pu|Ofe=pt)^Lq|Y;W!NHx^g>vq;3KVx@;%Y}dXRq*bSjL!5>4ce(73oP^q7}Hj|NkxfSwH%mS?)n1RnPpl19$bC0Sntu|yO3r)^avK9PkgmT4-E1SvW?fi4Leir_lQ>%^i>=n2=4nOL|fFA%JR*Q-dkO8q3Z<T_<)r#}Q}PA>'

def historical_embedded_records():
    dec=lambda b: pd.read_csv(io.BytesIO(zlib.decompress(base64.b85decode(b.encode('ascii')))))
    return dec(_HIST_TRAIN_B85),dec(_HIST_VALID_B85)
training_history, validation_history = historical_embedded_records()
print('Original logged training updates:',len(training_history),'| validation snapshots:',len(validation_history))
print('Range (updates):',int(validation_history.num_updates.min()),'to',int(validation_history.num_updates.max()))
fig,ax=plt.subplots(figsize=(10.5,4.2))
rolling=max(1,len(training_history)//200)
ax.plot(training_history.num_updates,training_history.loss.rolling(rolling,min_periods=1,center=True).mean(),label='Training (logged mini-batches)',lw=1.4)
ax.plot(validation_history.num_updates,validation_history.loss,label='Validation (recorded checkpoints)',lw=1.8)
ax.set(title='Original optimization history — training vs validation',xlabel='Optimizer updates',ylabel='Recorded objective loss')
ax.legend();ax.grid(alpha=.2);fig.tight_layout()
# Research record only: do NOT save into FIG_OUTPUT
plt.show()
print('Lowest archived validation loss:',float(validation_history.loss.min()),'at update',int(validation_history.loc[validation_history.loss.idxmin(),'num_updates']))


### 1b. Matched checkpoint evaluation: 10K training vs 10K validation — **RESEARCH RECORD ONLY**

Calculated from **existing** `objective_batches.csv` files. This is not an optimizer-step training curve. Retained for the study record; not included among slide-ready figures.


In [ ]:
if GENERAL:
    RUNS=GENERAL/'checkpoint_evaluation'
    trcsv=RUNS/'train_10k_batch8'/'objective_batches.csv'
    vacsv=RUNS/'full_10k_batch8'/'objective_batches.csv'
    a,b=read_csv_or_none(trcsv),read_csv_or_none(vacsv)
    if a is not None and b is not None:
        keys=[k for k in ['masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss','official_weighted_loss'] if k in a and k in b]
        df=pd.DataFrame({'Objective':keys,'Train mean':[a[k].mean() for k in keys],
                         'Validation mean':[b[k].mean() for k in keys]})
        df['Valid - train']=df['Validation mean']-df['Train mean']
        display(df.round(5))
        x=np.arange(len(df));fig,ax=plt.subplots(figsize=(11,4))
        ax.bar(x-.18,df['Train mean'],.36,label='Train 10K');ax.bar(x+.18,df['Validation mean'],.36,label='Validation 10K')
        ax.set_xticks(x,df['Objective'],rotation=20,ha='right');ax.set(ylabel='Original objective loss',title='Frozen checkpoint evaluation — matched sample sizes')
        ax.legend();plt.tight_layout();plt.show()
    else:print('Matched cached train/validation CSV unavailable; no evaluation will be run.')
section('Small train–validation loss gaps in sampled checkpoint evaluation. These are NOT optimizer-step training curves and cannot rule out all overfitting.')

## 2. Pretraining objective validation — presentation figure

Masked atom accuracy is **higher-is-better**. Coordinate RMSE (Å) is an **error metric — lower-is-better**. The 75.6% statistic is RMSE reduction, not percentage of conformers recovered. The chart is regenerated directly from saved JSON values.


In [ ]:
if GENERAL:
    RUNS=GENERAL/'checkpoint_evaluation'
    valid_json=RUNS/'full_10k_batch8'/'evaluation_summary.json'
    train_json=RUNS/'train_10k_batch8'/'evaluation_summary.json'
    val=json.loads(valid_json.read_text()) if valid_json.is_file() else {}
    tr=json.loads(train_json.read_text()) if train_json.is_file() else {}
    names={'n_molecules':'Molecules','masked_atom_accuracy':'Masked atom accuracy',
           'masked_atom_majority_class_baseline_accuracy':'Majority baseline',
           'coord_noisy_rmse':'Noisy coordinate RMSE (Å)',
           'coord_pred_rmse':'Recovered coordinate RMSE (Å)',
           'coord_rmse_reduction_fraction':'RMSE reduction fraction',
           'contrastive_top1_both_directions':'Batch top-1 (batch 8)',
           'positive_projection_cosine_mean':'Positive pair cosine'}
    if val:
        display(pd.DataFrame([{'Metric':label,'Train':tr.get(k,np.nan),'Validation':val.get(k,np.nan)}
                              for k,label in names.items()]).round(5))
        if val.get('coord_pred_rmse') is not None and val.get('coord_noisy_rmse'):
            print('Coordinate RMSE reduction: {:.1%}'.format(1-val['coord_pred_rmse']/val['coord_noisy_rmse']))
    else:print('Existing 10K evaluation_summary.json not found:',valid_json)
    if val and tr:
        fig,axes=plt.subplots(1,2,figsize=(11,4.15))
        xs=np.arange(2)
        train_a=100*float(tr.get('masked_atom_accuracy',np.nan));valid_a=100*float(val.get('masked_atom_accuracy',np.nan))
        train_b=100*float(tr.get('masked_atom_majority_class_baseline_accuracy',np.nan));valid_b=100*float(val.get('masked_atom_majority_class_baseline_accuracy',np.nan))
        axes[0].bar(xs-.18,[train_a,valid_a],.36,label='Pretrained Model')
        axes[0].bar(xs+.18,[train_b,valid_b],.36,label='Majority-class baseline')
        axes[0].set_xticks(xs,['Training 10K','Validation 10K'])
        axes[0].set(title='Masked atom reconstruction',ylabel='Accuracy (%)',ylim=(0,105))
        axes[0].legend(fontsize=8)
        noisy=float(val.get('coord_noisy_rmse',np.nan));recovered=float(val.get('coord_pred_rmse',np.nan))
        axes[1].bar(['Noisy input','Denoised output'],[noisy,recovered])
        axes[1].set(title='Validation 10K — 3D coordinate denoising',ylabel='Coordinate RMSE (Å) — lower is better')
        axes[1].text(.5,max(noisy,recovered)*.88,f'{(1-recovered/noisy)*100:.1f}% error reduction',ha='center',fontsize=9)
        fig.suptitle('General pretraining — atom and geometry validation')
        fig.tight_layout();export_fig(fig,'01_atom_and_geometry_validation')
    else:print('Missing saved train or validation summary: no chart is fabricated.')
else:print('Set GENERAL above to load cached 10K results.')
section('Strong reconstruction and coordinate denoising on the original objective; it does not establish RNA-delivery prediction or de-novo conformer generation.')

## 3. Per-element masked accuracy — presentation figure


In [ ]:
if GENERAL:
    atom_csv=RUNS/'full_10k_batch8'/'atom_token_class_accuracy.csv'
    atom=read_csv_or_none(atom_csv)
    if atom is not None:
        display(atom)
        required={'token_symbol','accuracy','masked_targets'}
        if required <= set(atom.columns):
            x=atom[atom.masked_targets >= 20].copy()
            fig,ax=plt.subplots(figsize=(10,4.2))
            ax.bar(x.token_symbol,x.accuracy*100)
            ax.set(title='Per-element masked atom recovery — validation 10K',ylabel='Accuracy (%)',ylim=(0,105))
            for j,r in enumerate(x.itertuples()):ax.text(j, getattr(r,'accuracy')*100+1,str(getattr(r,'masked_targets')),ha='center',fontsize=8)
            plt.tight_layout();export_fig(fig,'02_atom_accuracy_by_element')
    else:print('Missing cached atom CSV:',atom_csv)
section('C/N/O/S/F perform strongly; Br and I have much fewer masked targets and less reliable accuracy estimates. Overall 98% accuracy is dominated by common atoms.')

## 4. Molecular embedding PCA — chemistry-colored plots (no model inference)

Reproduce the original informative four-panel analysis: **CLS vs contrastive projection**, each colored by **heavy atom count** and **cLogP**. Uses **saved 10K embeddings and the saved validation sample manifest**. Lightweight RDKit property calculations from 10K saved SMILES are permitted; no LMDB reads or checkpoint inference. If the manifest or RDKit is missing, this section gives a clear diagnostic instead of producing an uninformative one-color PCA.

In [ ]:
if GENERAL:
    from sklearn.decomposition import PCA
    root = GENERAL/'checkpoint_evaluation'/'full_10k_batch8'
    meta_path = GENERAL/'validation_10k_seed17'/'sample_metadata.csv'
    if not meta_path.is_file():
        meta_path = find_file(GENERAL, 'sample_metadata.csv')
    print('Saved validation manifest:', meta_path or 'NOT FOUND')
    if meta_path is None or not meta_path.is_file():
        print('Missing saved SMILES manifest. No plain-color PCA substitution will be generated.')
    else:
        meta = pd.read_csv(meta_path, keep_default_na=False)
        if 'smiles' not in meta.columns:
            print('Manifest lacks smiles column; cannot restore chemical coloring.')
        else:
            try:
                from rdkit import Chem, RDLogger
                from rdkit.Chem import Descriptors
                RDLogger.DisableLog('rdApp.error')
                if not {'heavy_atoms','cLogP'}.issubset(meta.columns):
                    heavy,logp=[],[]
                    for smi in meta['smiles'].astype(str):
                        mol=Chem.MolFromSmiles(smi)
                        heavy.append(mol.GetNumHeavyAtoms() if mol is not None else np.nan)
                        logp.append(Descriptors.MolLogP(mol) if mol is not None else np.nan)
                    meta['heavy_atoms']=heavy
                    meta['cLogP']=logp
                    print('Descriptors derived from saved manifest SMILES (no LMDB/model rerun).')
                # Reuse original sample order where provided.
                if 'sample_order' in meta.columns:
                    meta=meta.sort_values('sample_order',kind='stable').reset_index(drop=True)
                paths=[('Pretrained CLS',root/'cls_view0.npy'),
                       ('Contrastive projection',root/'projected_view0.npy')]
                if not paths[1][1].is_file():
                    for alt in ['projector_view0.npy','projection_view0.npy']:
                        if (root/alt).is_file():paths[1]=('Contrastive projection',root/alt);break
                available=[]
                for label,path in paths:
                    if not path.is_file():
                        print('Cached embedding missing:',path.name);continue
                    arr=np.load(path,mmap_mode='r')
                    if arr.ndim!=2 or arr.shape[0]!=len(meta):
                        print('Row mismatch:',path.name,'embedding',arr.shape,'manifest',len(meta));continue
                    pca=PCA(n_components=2,svd_solver='randomized',random_state=17)
                    xy=pca.fit_transform(np.asarray(arr,dtype=np.float32))
                    available.append((label,xy,pca.explained_variance_ratio_))
                if available:
                    fig,axs=plt.subplots(len(available),2,figsize=(12.6,4.9*len(available)),squeeze=False)
                    for i,(label,xy,variance) in enumerate(available):
                        for j,feature in enumerate(['heavy_atoms','cLogP']):
                            ax=axs[i,j]
                            values=pd.to_numeric(meta[feature],errors='coerce').to_numpy()
                            mask=np.isfinite(values)
                            sc=ax.scatter(xy[mask,0],xy[mask,1],c=values[mask],s=3,
                                          alpha=.65,cmap='viridis',rasterized=True)
                            ax.set(title=f'{label} PCA — {feature}',
                                   xlabel=f'PC1 ({variance[0]:.1%})',
                                   ylabel=f'PC2 ({variance[1]:.1%})')
                            fig.colorbar(sc,ax=ax,label=feature)
                    fig.suptitle('Pretrained molecular representations — chemical property gradients',y=1.005)
                    fig.tight_layout();export_fig(fig,'03_pca_chemistry_colored_four_panels')
                    print('Source: cached 10K arrays + original saved manifest; PCA only.')
            except ImportError:
                print('RDKit is unavailable in this kernel; cannot calculate descriptor colors.')
else:
    print('Set GENERAL to the directory holding previous checkpoint outputs.')

## 5. Independent property benchmarks — presentation figures

Previously measured **three scaffold-disjoint splits** with 100% downstream labeled training data. Numeric results shown below are archived from the original analyses, not freshly refitted. The encoder was frozen. Some benchmark molecules overlap the general pretraining SMILES corpus.


In [ ]:
if GENERAL:
    CHEM=GENERAL/'chemistry_slide_evidence'
    print('Saved benchmark source:',CHEM,'exists:',CHEM.is_dir())
    print('No old benchmark PNGs are displayed (they retain outdated model labels).')
    # Archive values from the previously completed, 3-seed scaffold-disjoint benchmark.
    benchmark_archive={
      'ESOL':{'Pretrained Model':(.910,.031),'RDKit descriptors':(.981,.097),'Morgan-ECFP4':(1.713,.207),'Heavy atoms':(1.755,.198),'Mean label':(2.135,.199)},
      'FreeSolv':{'Pretrained Model':(2.350,.021),'RDKit descriptors':(1.828,.244),'Morgan-ECFP4':(2.726,.381),'Heavy atoms':(3.585,.557),'Mean label':(3.772,.501)},
      'Lipophilicity':{'Pretrained Model':(.924,.058),'RDKit descriptors':(1.015,.054),'Morgan-ECFP4':(.904,.087),'Heavy atoms':(1.181,.051),'Mean label':(1.192,.060)}
    }
    for name,results in benchmark_archive.items():
        labels=list(results);means=[results[k][0] for k in labels];sds=[results[k][1] for k in labels]
        idx=np.arange(len(labels));fig,ax=plt.subplots(figsize=(8.2,4.2))
        ax.barh(idx,means,xerr=sds,capsize=3)
        for ii,(mean,sd) in enumerate(zip(means,sds)):
            ax.text(mean+sd+max(means)*.015,ii,f'{mean:.3f} ± {sd:.3f}',va='center',fontsize=9)
        ax.set_xlim(0,max(np.array(means)+np.array(sds))*1.25)
        ax.set_yticks(idx,labels);ax.invert_yaxis()
        ax.set(xlabel='Scaffold-held-out RMSE (lower is better)',title=f'{name} — Pretrained Model vs chemical baselines')
        ax.grid(axis='x',alpha=.15);fig.tight_layout()
        export_fig(fig,'04_benchmark_'+name.lower())
    print('Archive metrics above are from previously saved results, NOT recomputed here.')

### 5a. Archived benchmark metric table

Read saved numerical results if available; otherwise display the archived measurements, explicitly labeled. This does not refit the probes.


In [ ]:
if GENERAL:
    if 'benchmark_archive' in globals():
        rows=[{'Benchmark':n,'Method':m,'RMSE mean':v[0],'RMSE SD':v[1]}
              for n,rs in benchmark_archive.items() for m,v in rs.items()]
        display(pd.DataFrame(rows))
    print('Source: previously completed 3-split benchmark (not newly fitted).')


### 5b. Additional ESOL / Lipophilicity evidence — from saved predictions and label-fraction results

The baseline RMSE bars above are **not the only benchmark evidence**. The original chemistry analysis may also contain (i) labeled-data efficiency (20% vs 100%) and (ii) predicted-vs-experimental scatter plots. This section searches **existing CSV result tables only**, recreates these figures under the neutral label **Pretrained Model**, and never refits a regression or runs inference. If a required column is missing, it prints the available tables rather than inventing values.

In [ ]:
if GENERAL:
    CHEM=GENERAL/'chemistry_slide_evidence'
    bench_root=CHEM/'independent_property_benchmarks'
    search_root=bench_root if bench_root.is_dir() else CHEM
    csvs=sorted(search_root.rglob('*.csv')) if search_root.is_dir() else []
    print('Cached chemistry CSV count:',len(csvs))
    # Be tolerant of alternate column names written by different benchmark versions.
    def matching_column(df,*terms):
        cols={str(x).lower().strip():x for x in df.columns}
        for term in terms:
            if term.lower() in cols:return cols[term.lower()]
        return None
    def neutral_label(x):
        t=str(x)
        if any(k in t.lower() for k in ['heart','frozen','pretrain']):return 'Pretrained Model'
        if 'ecfp' in t.lower() or 'morgan' in t.lower():return 'Morgan-ECFP4'
        if 'rdkit' in t.lower():return 'RDKit descriptors'
        return t
    frames=[]
    for p in csvs:
        try:
            df=pd.read_csv(p,nrows=300000)
            if len(df) and len(df.columns)>2:
                frames.append((p,df))
        except (OSError,UnicodeError,ValueError,pd.errors.ParserError):pass
    created=[]
    # Plot training label fraction using actual saved per-split benchmark rows.
    for bench in ['ESOL','Lipophilicity']:
        for p,df in frames:
            bcol=matching_column(df,'benchmark','dataset','task')
            mcol=matching_column(df,'model','method','representation')
            fcol=matching_column(df,'train_fraction','fraction','labeled_fraction','train_frac')
            rcol=matching_column(df,'rmse','test_rmse','scaffold_test_rmse')
            if mcol is None or fcol is None or rcol is None:continue
            subset=df.copy()
            if bcol is not None:
                subset=subset[subset[bcol].astype(str).str.lower()==bench.lower()]
            elif bench.lower() not in str(p).lower():continue
            if subset.empty:continue
            subset=subset.copy();subset['plot_method']=subset[mcol].map(neutral_label)
            subset['plot_fraction']=pd.to_numeric(subset[fcol],errors='coerce')
            subset['plot_rmse']=pd.to_numeric(subset[rcol],errors='coerce')
            subset=subset.dropna(subset=['plot_fraction','plot_rmse'])
            if subset.empty or subset.plot_fraction.nunique()<2:continue
            xscale=100 if subset.plot_fraction.max()<=1.0 else 1
            fig,ax=plt.subplots(figsize=(8,5))
            for method,grp in subset.groupby('plot_method',sort=False):
                agg=grp.groupby('plot_fraction').plot_rmse.agg(['mean','std']).sort_index()
                ax.errorbar(agg.index.to_numpy()*xscale,agg['mean'].to_numpy(),
                            yerr=agg['std'].fillna(0).to_numpy(),marker='o',capsize=3,label=method)
            ax.set(title=f'{bench} — labeled-data efficiency',
                   xlabel='Fraction of labeled training set (%)',
                   ylabel='Scaffold-held-out RMSE (lower is better)')
            ax.legend();ax.grid(alpha=.15);fig.tight_layout()
            export_fig(fig,'04_'+bench.lower()+'_labeled_data_efficiency');created.append(bench+' efficiency')
            print('Labeled-data source:',p);break
        else:
            print(f'{bench}: no cached per-fraction RMSE CSV found; no efficiency data fabricated.')
    # Held-out prediction scatter when cached prediction rows contain both true and predicted labels.
    for bench in ['ESOL','Lipophilicity']:
        for p,df in frames:
            true_col=matching_column(df,'y_true','true','experimental','target','label_true')
            pred_col=matching_column(df,'y_pred','prediction','predicted','pred','prediction_value')
            if true_col is None or pred_col is None:continue
            bcol=matching_column(df,'benchmark','dataset','task')
            mcol=matching_column(df,'model','method','representation')
            d=df.copy()
            if bcol is not None:d=d[d[bcol].astype(str).str.lower()==bench.lower()]
            elif bench.lower() not in str(p).lower():continue
            if mcol is not None:d=d[d[mcol].astype(str).map(neutral_label)=='Pretrained Model']
            if 'split' in d.columns:
                test=d[d['split'].astype(str).str.lower().isin(['test','heldout','held-out'])]
                if len(test):d=test
            if 'seed' in d.columns:
                seeds=pd.to_numeric(d['seed'],errors='coerce')
                if 17 in seeds.values:d=d[seeds==17]
            xx=pd.to_numeric(d[true_col],errors='coerce');yy=pd.to_numeric(d[pred_col],errors='coerce')
            keep=xx.notna()&yy.notna()
            if keep.sum()<8:continue
            xx=xx[keep].to_numpy();yy=yy[keep].to_numpy()
            lo=min(xx.min(),yy.min());hi=max(xx.max(),yy.max())
            fig,ax=plt.subplots(figsize=(6,5.5))
            ax.scatter(xx,yy,s=20,alpha=.65);ax.plot([lo,hi],[lo,hi],ls='--',color='gray')
            ax.set(title=f'{bench} — held-out predictions (Pretrained Model)',
                   xlabel='Experimental value',ylabel='Linear probe prediction')
            fig.tight_layout();export_fig(fig,'04_'+bench.lower()+'_prediction_scatter')
            created.append(bench+' prediction scatter');print('Prediction source:',p);break
        else:
            print(f'{bench}: no saved row-level prediction CSV found; scatter unavailable (not recomputed).')
    print('Extra figures created:',created or 'none; cached CSV schemas may be different')
else:print('General evidence directory not found.')

## 6. Structural retrieval and geometry examples — reference record

Older PNGs contain the original model label. For a consistent slide export, only inventory their cached paths below; no old figure is embedded or recomputed.


In [ ]:
if GENERAL:
    root=GENERAL/'chemistry_slide_evidence'
    for sub in ['structural_similarity','geometry_examples','slides_from_existing_10k']:
        d=root/sub
        if d.is_dir():
            pngs=sorted(d.rglob('*.png'))
            print(sub,':',len(pngs),'cached figures; original images preserved and not re-displayed')


## 7. Full-dataset unique/duplicate identity audit — saved results only

**DO NOT RERUN:** all unique molecule counts, 18.84M records, descriptor differences, scaffold bootstrap and conformer comparisons were already computed and saved on the HPC. This cell reads CSV/JSON exclusively.


In [ ]:
if DUPLICATE:
    print('Saved audit directory:',DUPLICATE)
    report=DUPLICATE/'expanded_report.json'
    if report.is_file():
        data=json.loads(report.read_text())
        print('Expanded report top-level fields:',list(data))
        for key,value in data.items():
            if isinstance(value,(int,float,str,bool)):
                print(f'{key}: {value}')
    else:print('Report JSON missing; existing saved figures may still be available.')
    for name in ['frequency_sample_summary.csv','scaffold_diversity.csv','duplicate_record_3d_pair_check.csv']:
        p=DUPLICATE/name
        if p.is_file():
            print('\nCached:',name)
            display(pd.read_csv(p).head(15))
    print('Existing full audit (read-only). The old overview PNG will NOT be embedded because its heading uses the old model name.')
    # Neutral reconstruction using already-established exact counts (no LMDB access).
    fig,ax=plt.subplots(figsize=(7.5,3.9))
    ax.bar(['Raw records','Distinct raw SMILES'],[18837320,13270476])
    ax.set(title='General ligand source — identity duplication',ylabel='Molecules / records')
    ax.text(.5,1.62e7,'29.55% excess records',ha='center',fontsize=11)
    fig.tight_layout();export_fig(fig,'05_dataset_duplicate_counts')
else:print('Duplicate evidence directory not found: set DUPLICATE in Section 0 to existing HPC path.')
section('18.84M raw records versus 13.27M unique raw SMILES (29.55% duplicate excess). Removing repeats changes exposure frequency; the evaluation below tests whether it materially changes measured chemical-space characteristics.')

## 8. Physicochemical distributions — existing unique/dedup analysis

Display cached comparison plots only where labels are generic. No descriptor calculations, bootstrap reruns, or file overwrites.


In [ ]:
if DUPLICATE:
    for stem in ['03_true_raw_vs_dedup_descriptor_distribution','04_descriptor_effect_sizes_bootstrap']:
        p=DUPLICATE/(stem+'.png')
        if p.is_file():show_image(p)
        else:print('Figure missing:',p.name)
    for name in ['descriptor_direct_comparison_bootstrap.csv','chemical_descriptor_reweight_preview.csv']:
        p=DUPLICATE/name
        if p.is_file():
            print('\nSaved measurements:',name)
            display(pd.read_csv(p).head(20))
section('Raw and unique-molecule samples have highly overlapping measured distributions (e.g. TPSA KS≈0.010; rotatable bonds KS≈0.006). Small descriptive shifts are not a formal statistical equivalence result.')

## 9. Scaffold diversity — existing equal-size comparisons (no recalculation)


In [ ]:
if DUPLICATE:
    p=DUPLICATE/'05_scaffold_diversity.png'
    if p.is_file():show_image(p)
    else:print('Missing saved scaffold figure:',p.name)
section('In equal-size n=3,000 samples, distinct Murcko scaffolds averaged about 2,731 (raw) versus 2,716 (dedup); effective diversity about 2,398 versus 2,360. No increase in sampled scaffold diversity was observed after deduplication.')

## 10. Duplicate-record geometry — cached evidence only


In [ ]:
if DUPLICATE:
    p=DUPLICATE/'06_3d_geometry_between_duplicate_records.png'
    if p.is_file():show_image(p)
    else:print('Missing saved geometry figure:',p.name)
    p=DUPLICATE/'duplicate_record_3d_pair_check.csv'
    if p.is_file():display(pd.read_csv(p).head(24))
section('24/24 sampled duplicate-record pairs had at least one near-identical coordinate slot at the exploratory 0.02 Å threshold. This does not prove all conformers across the pair are duplicated, nor generalize to all records.')

## 11. Interpretation, limitations, and next decision

**Supported by the existing saved evidence**

1. The completed general checkpoint shows strong masked-atom and coordinate-denoising performance on matched 10K validation samples.
2. Frozen CLS embeddings capture molecular variation and transfer to several external physicochemical property-label tasks, but do not uniformly outperform traditional RDKit/Morgan features.
3. The source contains considerable record-level SMILES duplication, yet the measured bulk descriptor distributions and equal-size scaffold-diversity comparisons change only modestly after deduplication.
4. The small sampled same-SMILES record-pair analysis provides preliminary evidence of duplicated 3D geometry.

**Important limits**

- These chemical-property benchmarks use scaffold-disjoint **labeled** splits, but some molecules have exact raw-SMILES overlap with the general pretraining corpus.
- Pretraining objective performance is **not** RNA/LNP delivery efficacy prediction.
- PCA, near-identical conformer pairs and bootstrap plots do **not** establish formal distributional equivalence or a downstream advantage from deduplicated pretraining.
- Current evidence supports **retaining the existing general checkpoint** and proceeding with the lipid-specific adaptation work; only a compute-matched ablation could establish performance effects of deduplication.

**Reproducibility policy:** All results above are loaded from saved HPC outputs. This notebook does not create raw-vs-unique comparisons, train probe models, run inference, or change data files.

## 12. Cached-file audit — what was located

In [ ]:
if GENERAL:
    print('GENERAL ROOT:',GENERAL)
    for sub in ['checkpoint_evaluation/full_10k_batch8/evaluation_summary.json',
                'checkpoint_evaluation/train_10k_batch8/evaluation_summary.json',
                'chemistry_slide_evidence/independent_property_benchmarks']:
        p=GENERAL/sub
        print('OK     ' if p.exists() else 'MISSING',p)
if DUPLICATE:
    print('DUPLICATE ROOT:',DUPLICATE)
    for name in ['expanded_report.json','00_PROGRESS_SLIDE_duplicate_impact.png',
                 '03_true_raw_vs_dedup_descriptor_distribution.png',
                 '05_scaffold_diversity.png','06_3d_geometry_between_duplicate_records.png']:
        p=DUPLICATE/name
        print('OK     ' if p.is_file() else 'MISSING',p)
print('Read-only summary complete. Nothing was rerun or written to the HPC source directories.')